# 07 - Second-dataset validation: LC25000 (lung subset)

**What this notebook does**: repeats the core of the MiniConvNet experiment on a *second*, completely
different dataset, so the paper's claim of "extensive experiments on two publicly available datasets"
is addressed rather than assumed. Trains MiniConvNet plus two baselines on the lung subset of
LC25000 and writes its own results table.

**This notebook is entirely additive.** It does not read, modify or re-run anything from notebooks
00-06. The CT-scan results (MiniConvNet 0.706 +/- 0.120 CV accuracy, all baselines valid) are final
and are not touched here. Every file this notebook writes is either new or `lc25000_`-prefixed.

---

## The dataset

**LC25000**, lung subset only - 3 classes, 5,000 768x768 histopathology images each:

| folder | class | description |
|---|---|---|
| `lung_aca` | `lung_adenocarcinoma` | malignant |
| `lung_scc` | `lung_squamous_cell_carcinoma` | malignant |
| `lung_n` | `lung_benign` | benign lung tissue |

**The colon classes are excluded** - `src/lc25000_utils.py` only ever looks up the three `lung_*`
folder names, so colon images are never indexed rather than being filtered out afterwards.

Download either published form; both work without configuration because the resolver searches for
those folder names at any depth:

* [andrewmvd/lung-and-colon-cancer-histopathological-images](https://www.kaggle.com/datasets/andrewmvd/lung-and-colon-cancer-histopathological-images)
* [doubleyouv10/lung-cancer-from-lc25000-dataset-split-10fold](https://www.kaggle.com/datasets/doubleyouv10/lung-cancer-from-lc25000-dataset-split-10fold)

Locally, put it in `Data_LC25000/`; on Kaggle, attach it and the resolver finds it.

**Note this is a different problem, not a harder version of the same one.** LC25000 is
histopathology (stained tissue at cellular scale); the primary dataset is CT slices (anatomical
scale). The class sets differ too - 3 vs 4, with no `large cell carcinoma` here. Numbers from the two
datasets are **not** directly comparable; what is comparable is the *shape* of the result, i.e.
whether detection is again far easier than subtype discrimination (LESSON 11).

---

## Disclosed scope reduction

Same pattern used throughout this project: reduce deliberately, state it plainly, never let a reader
mistake a reduced experiment for a full one.

> **[CHOICE] One training run, not cross-validation.** The primary experiment reports a 3-fold CV
> mean +/- std because a single favourable split had previously misled us (LESSON 10). Here a single
> clean run is enough to report a second-dataset number, and CV would triple the CPU cost of a
> secondary result. **The LC25000 number therefore has no error bar and must never be quoted beside
> the CT number as though both were cross-validated.**

> **[CHOICE] Two baselines, not four.** `MobileNetV3Small` (lightweight) and `ResNet50` (heavy) span
> the range; VGG16 and EfficientNetV2B0 add cost without adding contrast. Both use the same frozen-
> backbone, head-only protocol as the primary experiment.

> **[CHOICE] Stratified subsample, decided by the timing gate.** LC25000's lung subset is 15,000
> images against the CT dataset's ~1,000. Section 3 measures one real epoch and, if the projection
> exceeds the project's 20-minute threshold, the run uses a stratified subsample
> (`LC25000_IMAGES_PER_CLASS`, default 1,200/class = 3,600 images). The sample size is recorded in
> the results row, so a reduced run can never be read as a full-dataset one.

**Everything else is unchanged from the validated primary setup** and is not re-derived here: the
LeakyReLU + He-init + wide-bottleneck architecture, `Adam(lr=1e-4, clipnorm=1.0)`, label smoothing,
`ReduceLROnPlateau`, both collapse checks, and raw prediction saving.

**What "looks right"**: LC25000 is large, well-balanced and visually much easier than CT, so expect
high accuracy (published work reports high-90s) and **no** collapse. Both checks still run - a
well-balanced dataset makes collapse unlikely, not impossible, and the point of LESSON 4 is that the
check is automatic rather than conditional on expectations.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.lc25000_utils import (LC25000_CLASS_NAMES, LC25000_NUM_CLASSES,
                               LC25000_IMAGES_PER_CLASS, LC25000_RESULTS_CSV,
                               resolve_lc25000_root, init_lc25000_results)

ensure_dirs()
print('classes        :', LC25000_CLASS_NAMES)
print('results table  :', LC25000_RESULTS_CSV.name, '(separate from the CT results_table.csv)')
print('created        :', init_lc25000_results())
print()
print('LC25000 root   :', resolve_lc25000_root())

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

# Reused unchanged from the validated primary pipeline - both builders already
# take num_classes, and nothing in train_utils is class-count specific.
from src.models import build_miniconvnet, build_baseline, count_params
from src.train_utils import (set_global_seeds, compute_report, compile_model, optimizer_summary,
                             compute_class_weights, make_callbacks, make_epoch_timer,
                             save_history, plot_history, final_epoch_summary,
                             estimate_training_time)
from src.evaluate_utils import print_collapse_report

# 3-class-aware layer, all of it new and additive.
from src.lc25000_utils import (index_lc25000, lc25000_class_counts, check_duplicate_filenames,
                               stratified_subsample, build_lc25000_split, lc25000_split_counts,
                               save_lc25000_split, make_lc25000_datasets, lc25000_metrics,
                               lc25000_detect_collapse, lc25000_confusion,
                               lc25000_per_class_report, plot_lc25000_confusion,
                               benign_vs_subtype_breakdown, interpret_lc25000_breakdown,
                               save_lc25000_predictions, load_lc25000_predictions,
                               record_lc25000_result, load_lc25000_results, lc25000_result_row)

set_global_seeds(SEED)
for k, v in compute_report().items():
    print(f'{k}: {v}')

## 1. Index the lung images

**Looks right**: 5,000 images per class, 15,000 total, and three folders - `lung_aca`, `lung_n`,
`lung_scc`. If a colon folder appears in the `folder` column something is wrong with the resolver;
it should be impossible.

In [ ]:
df = index_lc25000()
print('images indexed:', len(df))
print()
print(lc25000_class_counts(df).to_string())
print()
print('source folders found:', sorted(df['folder'].unique()))
assert not any('colon' in f.lower() for f in df['folder'].unique()), 'colon images leaked in'
print('colon exclusion: PASSED')
print()
print(df[['filename', 'class', 'label', 'folder']].head(3).to_string(index=False))

## 2. Data caveat, stated rather than checked away

LC25000 is generated by augmenting **250 original images per class** up to 5,000 - so near-duplicate
images exist *by construction*, and a random train/test split will place augmentations of the same
source image on both sides. That is a property of the dataset, and it is the main reason published
LC25000 accuracies run so high.

The primary CT experiment builds a content-hash-deduplicated `clean` split to quantify exactly this
(LESSON 6). **That is not repeated here** - it would need a source-image grouping the dataset does
not publish, and this is a secondary experiment. The check below is filename-level only; the caveat
is carried into the README instead of being silently dropped.

In [ ]:
dup = check_duplicate_filenames(df)
for k, v in dup.items():
    print(f'{k}: {v}')

## 3. Sample, split, and estimate the cost before training anything

The sampling decision is made **after** measuring, not guessed. The gate is the same one used
everywhere in this project: measure one real epoch, extrapolate, stop and ask above 20 minutes.

In [ ]:
# Start from the subsample; section 3b re-checks whether the full set is affordable.
USE_FULL_DATASET = False     # flip to True only if the timing gate below says it is affordable

sampled = df if USE_FULL_DATASET else stratified_subsample(df, per_class=LC25000_IMAGES_PER_CLASS)
dropped = len(df) - len(sampled)
print(f'using {len(sampled)} of {len(df)} images'
      + ('' if USE_FULL_DATASET else f' ({dropped} not sampled - disclosed scope reduction)'))
print()
print(lc25000_class_counts(sampled).to_string())

split = build_lc25000_split(sampled)
print()
print(lc25000_split_counts(split).to_string())
print('\nsplit definition saved to', save_lc25000_split(split))

In [ ]:
# one_hot=True because the model is compiled with label smoothing, exactly as in
# the primary experiment - but 3-wide here, which is why the CT make_dataset is
# not reused.
train_ds, val_ds, test_ds, frames = make_lc25000_datasets(split, one_hot=True)

x, y = next(iter(train_ds))
print('image batch:', x.shape, x.dtype)
print('label batch:', y.shape, '(one-hot, 3 wide)')
print(f'pixel range: min={float(tf.reduce_min(x)):.1f} max={float(tf.reduce_max(x)):.1f} '
      '(raw [0,255]; the model rescales internally)')
assert y.shape[-1] == LC25000_NUM_CLASSES, 'targets are not 3-wide'
print('\ntrain/val/test sizes:', {k: len(v) for k, v in frames.items()})

In [ ]:
# 3b. Timing gate. MiniConvNet is timed; the baselines get their own estimate later.
est_mini = estimate_training_time(
    model_fn=lambda: compile_model(build_miniconvnet(num_classes=LC25000_NUM_CLASSES),
                                   verbose=False),
    train_ds=train_ds, val_ds=val_ds,
    planned_epochs=EPOCHS_MINICONVNET, n_runs=1)
print()
if est_mini['exceeds_threshold']:
    print('Over the threshold. Options, in order of preference:')
    print('  1. lower LC25000_IMAGES_PER_CLASS in src/lc25000_utils.py and re-run section 3;')
    print('  2. lower EPOCHS_MINICONVNET for this notebook only;')
    print('  3. agree the longer run explicitly before starting section 4.')
    print('Do NOT just run it - the whole point of this gate is that the cost is agreed first.')
else:
    print('Within budget - proceed to section 4.')

## 4. MiniConvNet on LC25000

The same ~0.5M architecture as the primary experiment, with `num_classes=3`. The only difference is
the output layer width, so the parameter count drops slightly (the 64-unit bottleneck feeds 3 units
instead of 4): **499,107** instead of 499,172.

**Looks right**: all three classes predicted, `status = ok`, and accuracy far above the 0.333 chance
level - LC25000 is a much easier problem than the CT dataset.

In [ ]:
def train_and_evaluate_lc25000(model, run_name, arch_variant, epochs, datasets,
                               class_weight=None, note=''):
    """Train, evaluate, collapse-check, save predictions, record.

    ``datasets`` is an explicit ``(train_ds, val_ds, test_ds)`` triple: MiniConvNet
    uses the one-hot views (label smoothing) and the baselines use the sparse ones,
    so the runner must never reach for a global.
    """
    fit_train, fit_val, fit_test = datasets
    print('=' * 72)
    print('run:', run_name)
    print('params   :', count_params(model)['total_params'])
    print('optimizer:', optimizer_summary(model))

    timer = make_epoch_timer(verbose=1)
    history = model.fit(fit_train, validation_data=fit_val, epochs=epochs,
                        class_weight=class_weight,
                        callbacks=make_callbacks(f'lc25000_{run_name}', timer=timer),
                        verbose=2)
    save_history(history, f'lc25000_{run_name}', timer=timer)
    summary = final_epoch_summary(history, timer=timer)
    print()
    for k, v in summary.items():
        print(f'  {k}: {v}')

    y_prob = model.predict(fit_test, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = np.concatenate([np.asarray(t) for _, t in fit_test.as_numpy_iterator()])
    if y_true.ndim > 1:
        y_true = np.argmax(y_true, axis=1)
    y_true = y_true.astype(int)
    assert len(y_true) == len(y_pred), 'label/prediction length mismatch'

    # Evidence first (LESSON 5), conclusions second.
    save_lc25000_predictions(run_name, y_true, y_pred, y_prob,
                             meta={'arch_variant': arch_variant, 'note': note,
                                   'n_train': len(frames['train']),
                                   'sampled_per_class': None if USE_FULL_DATASET
                                                        else LC25000_IMAGES_PER_CLASS})

    metrics = lc25000_metrics(y_true, y_pred, y_prob)
    collapse = lc25000_detect_collapse(history=history, kappa=metrics['cohen_kappa'],
                                       mcc=metrics['mcc'], y_pred=y_pred)
    breakdown = benign_vs_subtype_breakdown(y_true, y_pred)

    print('\ntest metrics:')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}')
    print('\nconfusion matrix (an all-zero COLUMN = a class never predicted):')
    print(lc25000_confusion(y_true, y_pred))
    print()
    print_collapse_report(collapse, f'lc25000_{run_name}')
    print('\nper-class report:')
    print(lc25000_per_class_report(y_true, y_pred).round(4))
    print('\n' + interpret_lc25000_breakdown(breakdown))

    plot_history(history, f'lc25000_{run_name}')
    plot_lc25000_confusion(y_true, y_pred, run_name)

    record_lc25000_result(lc25000_result_row(
        model_name=run_name, metrics=metrics, collapse=collapse, breakdown=breakdown,
        params=count_params(model)['total_params'], arch_variant=arch_variant,
        n_train=len(frames['train']), n_test=len(frames['test']),
        epochs_trained=summary['epochs_trained'],
        notes=(f"single run, no CV (disclosed scope reduction); "
               f"{'full dataset' if USE_FULL_DATASET else f'stratified subsample {LC25000_IMAGES_PER_CLASS}/class'}; "
               f"{summary.get('mean_seconds_per_epoch')}s/epoch, "
               f"{summary.get('total_minutes')} min on CPU{note}")))

    tf.keras.backend.clear_session()
    return {'run_name': run_name, 'metrics': metrics, 'collapse': collapse,
            'breakdown': breakdown, 'summary': summary,
            'n_predicted_classes': collapse['details'].get('n_predicted_classes'),
            'params': count_params(model)['total_params']}

In [ ]:
set_global_seeds(SEED)
mini = build_miniconvnet(num_classes=LC25000_NUM_CLASSES)
mini.summary()
print()
print('params:', count_params(mini))
print('(the CT build is 499,172; the difference is the 3-unit vs 4-unit output layer)')
compile_model(mini)

In [ ]:
results = {}
results['MiniConvNet'] = train_and_evaluate_lc25000(
    mini, 'miniconvnet', arch_variant='miniconvnet_500k', epochs=EPOCHS_MINICONVNET,
    datasets=(train_ds, val_ds, test_ds))

## 5. Two baselines, frozen backbones

`MobileNetV3Small` and `ResNet50`, same feature-extraction protocol as the primary experiment
(backbone frozen, head-only training, sparse labels and no label smoothing - smoothing is a
MiniConvNet anti-collapse measure, not part of the baseline protocol).

Timed first, as always.

In [ ]:
# Baselines take sparse labels, so build a second view of the same split.
from src.lc25000_utils import make_lc25000_dataset

b_train = make_lc25000_dataset(frames['train'], shuffle=True, augment=True, seed=SEED)
b_val = make_lc25000_dataset(frames['val'])
b_test = make_lc25000_dataset(frames['test'])
print('baseline datasets built (sparse labels)')

est_base = estimate_training_time(
    model_fn=lambda: compile_model(build_baseline('ResNet50', num_classes=LC25000_NUM_CLASSES),
                                   lr=LR_BASELINE, label_smoothing=0.0, verbose=False),
    train_ds=b_train, val_ds=b_val,
    planned_epochs=EPOCHS_BASELINE, n_runs=2)
print()
print('Scaled from ResNet50; MobileNetV3Small is much faster, so this over-estimates the pair.')

In [ ]:
# Frozen-backbone baseline on LC25000, using the sparse-label datasets.
def run_lc25000_baseline(name, epochs=EPOCHS_BASELINE):
    set_global_seeds(SEED)
    model = build_baseline(name, num_classes=LC25000_NUM_CLASSES, trainable_base=False)
    compile_model(model, lr=LR_BASELINE, label_smoothing=0.0)
    return train_and_evaluate_lc25000(
        model, name.lower(), arch_variant='transfer_frozen', epochs=epochs,
        datasets=(b_train, b_val, b_test),
        note='; ImageNet feature extraction, head-only training')

In [ ]:
results['MobileNetV3Small'] = run_lc25000_baseline('MobileNetV3Small')

In [ ]:
results['ResNet50'] = run_lc25000_baseline('ResNet50')

## 6. Results and cross-dataset comparison

**Looks right**: three rows, all `ok`, all predicting 3/3 classes.

The comparison that matters is **not** LC25000 accuracy against CT accuracy - different problems,
different class sets, different imaging modality. It is whether the *detection-vs-subtyping* gap that
dominated the CT results (LESSON 11: detection 92-99%, subtyping 35-66% for every model tested)
appears here too, or whether it was specific to CT.

In [ ]:
tbl = pd.DataFrame([
    {'model': k,
     'params': v['params'],
     'accuracy': round(v['metrics']['accuracy'], 4),
     'f1_macro': round(v['metrics']['f1_macro'], 4),
     'cohen_kappa': round(v['metrics']['cohen_kappa'], 4),
     'benign_vs_malignant': round(v['breakdown']['benign_vs_malignant_accuracy'], 4),
     'subtype_acc': round(v['breakdown']['subtype_accuracy_all_malignant'], 4),
     'classes_pred': v['n_predicted_classes'],
     'epochs': v['summary']['epochs_trained'],
     'minutes': v['summary'].get('total_minutes'),
     'status': v['collapse']['status']}
    for k, v in results.items()])
tbl['detection_minus_subtype'] = (tbl['benign_vs_malignant'] - tbl['subtype_acc']).round(4)
print(tbl.to_string(index=False))

bad = tbl[tbl['status'] != VALID_TAG]
print('\ninvalid runs:', bad['model'].tolist() if len(bad) else 'none - all three valid')

In [ ]:
print('LESSON 11 across both datasets - detection vs subtype discrimination')
print('=' * 72)
print('CT dataset (primary, from outputs/results_table.csv):')
print('  every model detected tumour at 92-99% but told subtypes apart at only 35-66%.')
print()
print('LC25000 (this notebook):')
for _, r in tbl.iterrows():
    print(f"  {r['model']:18s} detection={r['benign_vs_malignant']:.4f} "
          f"subtype={r['subtype_acc']:.4f} gap={r['detection_minus_subtype']:+.4f}")
print()
print('If the gap is large here too, subtype discrimination is the hard part of the task in')
print('general, not an artefact of the CT dataset. If it is small, the CT gap was dataset-specific')
print('- which is just as informative. Write whichever the numbers actually support.')

In [ ]:
print('outputs/results_table_lc25000.csv')
print(load_lc25000_results().to_string(index=False))
print()
print('Prediction files written (all lc25000_-prefixed, none overwriting a CT run):')
for name in results:
    stem = f"lc25000_{name.lower() if name != 'MiniConvNet' else 'miniconvnet'}"
    print(' ', PREDICTIONS_DIR / f'{stem}_predictions.csv')
print()
print('Nothing in outputs/results_table.csv, experiments_log.csv or ablation_dropout.csv was '
      'touched by this notebook.')
print('\nNext: copy these numbers into the README\'s "Second-dataset validation (LC25000)" '
      'section - do not hand-write numbers the pipeline did not produce.')